# Model 2 — Does what is learned in one country work in another? (UK MOT vs Dutch RDW)  (experiments only)

**Question.** The Israeli Ministry of Transport does not publish inspection outcomes, so a Model 2 for Israeli plates would be trained abroad. Before choosing between the UK and the Dutch data, this notebook measures
whether a model trained on one country predicts defects in the other, using only features that exist in both datasets **and** in the Israeli registry.

**Design.**
* Same outcome definitions as notebooks 04/05 (defect found at an inspection, by system). The *definitions are not identical* (UK groups by test-item section; NL by regulation article): brakes are very close, lamps/electrical and emissions/leaks are close but not the same. So absolute rates differ; the experiment is about **ranking** (AUC, capture) and about **what shifts** (calibration).
* Common feature set: **age in whole years** (calendar year of the test minus year of first use, exactly what the Israeli registry's year of manufacture gives), **fuel** (petrol / diesel / hybrid), **make** (only makes that have at least 300 rows in *both* countries; the rest are `OTHER`), and optionally **model name** (only names with at least 100 rows in both).
* Common population: age 4-20 years, petrol/diesel/hybrid (electric and gas are not comparable: 2,130 UK electric rows only).
* Every number is computed on a held-out set of **vehicles never seen in training** (30 % of cars/plates per country). In-country and foreign models are scored on exactly the same test rows.
* Calibration under country shift is reported both raw and after a single-number base-rate correction; the second needs the destination's overall defect rate, which for Israel would have to come from the Ministry (not shown to exist as open data).

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.options.display.width = 190; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.4f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import INTERIM_DIR
from real_outcome_data import load_both
from israel_registry import normalise_registry
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
from scipy.optimize import brentq
from scipy.stats import spearmanr
try:
    display
except NameError:
    display = print
uk, nl = load_both()
print(f"UK: {len(uk):,} tests / {uk.unit.nunique():,} vehicles | NL: {len(nl):,} inspections / {nl.unit.nunique():,} cars")

UK: 370,633 tests / 370,460 vehicles | NL: 812,021 inspections / 371,250 cars


## 1. Common population and common vocabulary

In [2]:
AGE, FUELS = (4, 20), ["petrol", "diesel", "hybrid"]
def compat(d): return d[d.age_year.between(*AGE) & d.fuel.isin(FUELS)].copy()
uk, nl = compat(uk), compat(nl)
# Opel (Israel, NL) is sold as Vauxhall in the UK: same cars, one make
uk["make"] = uk.make.replace({"Vauxhall": "Opel"}); uk["model"] = uk.model.str.replace(r"^Vauxhall ", "Opel ", regex=True)
mk_uk, mk_nl = uk.make.value_counts(), nl.make.value_counts()
COMMON_MAKES = sorted(set(mk_uk[mk_uk >= 300].index) & set(mk_nl[mk_nl >= 300].index))
md_uk, md_nl = uk.model.value_counts(), nl.model.value_counts()
COMMON_MODELS = sorted(set(md_uk[md_uk >= 100].index) & set(md_nl[md_nl >= 100].index))
for d in (uk, nl):
    d["make_c"] = d.make.where(d.make.isin(COMMON_MAKES), "OTHER")
    d["model_c"] = d.model.where(d.model.isin(COMMON_MODELS), "OTHER")
print(f"common population: UK {len(uk):,} tests, NL {len(nl):,} inspections")
print(f"{len(COMMON_MAKES)} makes appear in both countries with enough rows; share of rows with a common make: UK {(uk.make_c != 'OTHER').mean():.3f}, NL {(nl.make_c != 'OTHER').mean():.3f}")
print(f"{len(COMMON_MODELS)} model names in both; share of rows with a common model: UK {(uk.model_c != 'OTHER').mean():.3f}, NL {(nl.model_c != 'OTHER').mean():.3f}")
print("prevalence per test/inspection:")
display(pd.DataFrame({c: d[["brakes", "electrical", "engine_emissions"]].mean() for c, d in (("UK", uk), ("NL", nl))}).T)
print("fuel mix:", {c: d.fuel.value_counts(normalize=True).round(3).to_dict() for c, d in (("UK", uk), ("NL", nl))})
print("median age:", {c: float(d.age_year.median()) for c, d in (("UK", uk), ("NL", nl))})

common population: UK 340,371 tests, NL 643,585 inspections
33 makes appear in both countries with enough rows; share of rows with a common make: UK 0.984, NL 0.968


166 model names in both; share of rows with a common model: UK 0.616, NL 0.530
prevalence per test/inspection:


,brakes,electrical,engine_emissions
UK,0.0677,0.1134,0.0305
NL,0.0799,0.1813,0.0261


fuel mix: {'UK': {'petrol': 0.527, 'diesel': 0.449, 'hybrid': 0.024}, 'NL': {'petrol': 0.859, 'diesel': 0.095, 'hybrid': 0.046}}
median age: {'UK': 9.0, 'NL': 9.0}


## 2. How well does each country's fleet cover the *Israeli* fleet? (what a model could ever be asked about)
A random sample of 19,200 Israeli passenger registrations (same government resource the backend queries) is put through the same vocabulary.

In [3]:
il = normalise_registry(pd.read_csv(INTERIM_DIR / "gov_registry_sample.csv"))
il = il[il.fuel.notna()].copy()
il["age_year"] = il.age.astype(int)
IL_ALL = len(il)
il["in_pop"] = il.age_year.between(*AGE) & il.fuel.isin(FUELS)
print(f"Israeli passenger sample: {IL_ALL:,}. Inside the common population (age {AGE[0]}-{AGE[1]}, petrol/diesel/hybrid): {il.in_pop.mean():.3f}")
print("outside it:", {"age<4": float((il.age_year < AGE[0]).mean()), "age>20": float((il.age_year > AGE[1]).mean()), "electric / gas": float((~il.fuel.isin(FUELS)).mean())})
ilp = il[il.in_pop].copy()
ilp["make"] = ilp.make.str.title()                     # same capitalisation as the UK/NL tables (BMW -> Bmw, MG -> Mg)
ilp["model"] = ilp.make + " " + ilp.model_raw.astype(str).str.title()
cov = pd.DataFrame({
    "share of Israeli vehicles whose make is in the UK data (>=300 rows)": [float(ilp.make.isin(mk_uk[mk_uk >= 300].index).mean())],
    "... in the NL data (>=300 rows)": [float(ilp.make.isin(mk_nl[mk_nl >= 300].index).mean())],
    "... in both (the common make list)": [float(ilp.make.isin(COMMON_MAKES).mean())],
    "share whose make+model name is in UK data (>=100 rows)": [float(ilp.model.isin(md_uk[md_uk >= 100].index).mean())],
    "... in NL data": [float(ilp.model.isin(md_nl[md_nl >= 100].index).mean())],
}).T.rename(columns={0: "value"})
display(cov)
print("Israeli makes NOT in the common list, by share of the Israeli fleet:")
display((ilp[~ilp.make.isin(COMMON_MAKES)].make.value_counts() / len(ilp)).head(12).rename("share of ALL Israeli vehicles in the population"))

Israeli passenger sample: 19,200. Inside the common population (age 4-20, petrol/diesel/hybrid): 0.663
outside it: {'age<4': 0.30255208333333333, 'age>20': 0.019947916666666666, 'electric / gas': 0.07140625}


,value
share of Israeli vehicles whose make is in the UK data (>=300 rows),0.9786
... in the NL data (>=300 rows),0.9893
... in both (the common make list),0.9755
share whose make+model name is in UK data (>=100 rows),0.5929
... in NL data,0.5225


Israeli makes NOT in the common list, by share of the Israeli fleet:


make
Daihatsu   0.0107
Unmapped   0.0070
Chrysler   0.0031
Mg         0.0031
Chery      0.0006
Name: share of ALL Israeli vehicles in the population, dtype: float64

## 3. Cross-country transfer matrix
Train on UK, on NL, or on both pooled (no country flag); test on the UK test cars and on the NL test cars. XGBoost with the natural objective (as in notebooks 04/05).

In [4]:
def split(d, seed=0):
    a, b = next(GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=seed).split(d, groups=d.unit))
    assert not set(d.iloc[a].unit) & set(d.iloc[b].unit)
    return d.iloc[a].copy(), d.iloc[b].copy()
uk_tr, uk_te = split(uk); nl_tr, nl_te = split(nl)
CAT = {"fuel": FUELS, "make_c": COMMON_MAKES + ["OTHER"], "model_c": COMMON_MODELS + ["OTHER"]}
def prep(df, cols):
    X = df[cols].copy()
    for c in cols:
        if c in CAT: X[c] = pd.Categorical(X[c], categories=CAT[c])
    return X
FEATS = {"age only": ["age_year"], "age, fuel, make": ["age_year", "fuel", "make_c"], "age, fuel, make, model": ["age_year", "fuel", "make_c", "model_c"]}
TARGETS = {"brakes": "brakes", "electrical": "electrical", "engine-related": "engine_emissions"}
def fit_xgb(X, y):
    return XGBClassifier(n_estimators=250, max_depth=6, learning_rate=0.06, subsample=0.8, colsample_bytree=0.8, min_child_weight=50, tree_method="hist",
                         enable_categorical=True, n_jobs=8, random_state=0).fit(X, y)
def capture(y, p, frac=0.2):
    k = int(len(p) * frac); idx = np.argsort(-p)[:k]; y = np.asarray(y); return float(y[idx].sum() / y.sum())
def ece(y, p, bins=10):
    q = pd.qcut(p, bins, duplicates="drop"); t = pd.DataFrame({"y": y, "p": p}).groupby(q, observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return float((abs(t.p - t.y) * t.n).sum() / t.n.sum())
def shift_to_rate(p, rate):
    lg = np.log(np.clip(p, 1e-9, 1 - 1e-9) / (1 - np.clip(p, 1e-9, 1 - 1e-9)))
    f = lambda dl: (1 / (1 + np.exp(-(lg + dl)))).mean() - rate
    return 1 / (1 + np.exp(-(lg + brentq(f, -6, 6))))
def boot_auc(y, p, units, n=60, seed=0):
    rng = np.random.default_rng(seed); u = pd.Series(np.arange(len(y))).groupby(units.to_numpy()).apply(lambda s: s.to_numpy()); keys = u.index.to_numpy(); out = []
    for _ in range(n):
        ii = np.concatenate(u.loc[rng.choice(keys, len(keys))].to_list()); out.append(roc_auc_score(y[ii], p[ii]))
    return np.percentile(out, [2.5, 97.5])
TRAIN = {"UK": uk_tr, "NL": nl_tr, "UK+NL pooled": pd.concat([uk_tr, nl_tr])}
TEST = {"UK": uk_te, "NL": nl_te}
rows, PRED = [], {}
for tname, tcol in TARGETS.items():
    for fname, cols in FEATS.items():
        for trn, trd in TRAIN.items():
            m = fit_xgb(prep(trd, cols), trd[tcol])
            for ten, ted in TEST.items():
                p = m.predict_proba(prep(ted, cols))[:, 1]; y = ted[tcol].to_numpy(); PRED[(tname, fname, trn, ten)] = p
                lo, hi = boot_auc(y, p, ted.unit)
                rows.append({"system": tname, "features": fname, "trained on": trn, "tested on": ten, "ROC-AUC": roc_auc_score(y, p), "CI low": lo, "CI high": hi,
                             "capture top 20%": capture(y, p), "observed rate": y.mean(), "mean predicted": p.mean(),
                             "ECE raw": ece(y, p), "ECE after rate correction": ece(y, shift_to_rate(p, y.mean()))})
R = pd.DataFrame(rows)
display(R[R.features == "age, fuel, make"].drop(columns=["features"]))

,system,trained on,tested on,ROC-AUC,CI low,CI high,capture top 20%,observed rate,mean predicted,ECE raw,ECE after rate correction
6,brakes,UK,UK,0.6766,0.6719,0.6804,0.3895,0.0668,0.0679,0.0033,0.0034
7,brakes,UK,NL,0.6488,0.6437,0.6524,0.3538,0.0812,0.0726,0.0118,0.0094
8,brakes,NL,UK,0.6612,0.6559,0.6653,0.3732,0.0668,0.0751,0.0083,0.0043
9,brakes,NL,NL,0.6653,0.6610,0.6684,0.3829,0.0812,0.0795,0.0028,0.0027
10,brakes,UK+NL pooled,UK,0.6730,0.6682,0.6766,0.3871,0.0668,0.0711,0.0047,0.0031
11,brakes,UK+NL pooled,NL,0.6642,0.6598,0.6677,0.3800,0.0812,0.0777,0.0037,0.0031
24,electrical,UK,UK,0.7099,0.7067,0.7140,0.4013,0.1130,0.1132,0.0035,0.0035
25,electrical,UK,NL,0.6678,0.6648,0.6717,0.3467,0.1843,0.1206,0.0637,0.0191
26,electrical,NL,UK,0.7020,0.6984,0.7060,0.3977,0.1130,0.1724,0.0594,0.0083
27,electrical,NL,NL,0.6765,0.6737,0.6803,0.3563,0.1843,0.1800,0.0045,0.0027


### 3b. The same, summarised: in-country vs foreign model on the same test cars
`gap` = in-country AUC minus foreign-trained AUC (positive = something country-specific is lost).

In [5]:
rows = []
for tname in TARGETS:
    for fname in FEATS:
        for ten, other in (("UK", "NL"), ("NL", "UK")):
            q = R[(R.system == tname) & (R.features == fname) & (R["tested on"] == ten)].set_index("trained on")
            rows.append({"system": tname, "features": fname, "tested on": ten, "in-country AUC": q.loc[ten, "ROC-AUC"], f"trained on {other}": q.loc[other, "ROC-AUC"],
                         "gap": q.loc[ten, "ROC-AUC"] - q.loc[other, "ROC-AUC"], "pooled UK+NL": q.loc["UK+NL pooled", "ROC-AUC"],
                         "capture in-country": q.loc[ten, "capture top 20%"], "capture foreign": q.loc[other, "capture top 20%"]})
G = pd.DataFrame(rows); display(G)

,system,features,tested on,in-country AUC,trained on NL,gap,pooled UK+NL,capture in-country,capture foreign,trained on UK
0,brakes,age only,UK,0.6547,0.6548,-0.0001,0.6548,0.3604,0.3604,NaN
1,brakes,age only,NL,0.6532,NaN,-0.0000,0.6532,0.3606,0.3606,0.6532
2,brakes,"age, fuel, make",UK,0.6766,0.6612,0.0154,0.6730,0.3895,0.3732,NaN
3,brakes,"age, fuel, make",NL,0.6653,NaN,0.0166,0.6642,0.3829,0.3538,0.6488
4,brakes,"age, fuel, make, model",UK,0.6809,0.6522,0.0287,0.6779,0.3959,0.3705,NaN
5,brakes,"age, fuel, make, model",NL,0.6686,NaN,0.0252,0.6677,0.3847,0.3420,0.6433
6,electrical,age only,UK,0.6857,0.6852,0.0005,0.6852,0.3692,0.3692,NaN
7,electrical,age only,NL,0.6541,NaN,0.0000,0.6541,0.3384,0.3382,0.6540
8,electrical,"age, fuel, make",UK,0.7099,0.7020,0.0079,0.7056,0.4013,0.3977,NaN
9,electrical,"age, fuel, make",NL,0.6765,NaN,0.0087,0.6754,0.3563,0.3467,0.6678


## 4. Is the transferred signal the same *kind* of signal? Age curves and make effects, country against country
If a model transfers because both countries share the same physical wear pattern, then (a) risk should rise with age in the same shape and (b) makes that are above average in one country should be above average in the other, after removing the age/fuel mix.

In [6]:
rows = []
for tcol in ["brakes", "electrical", "engine_emissions"]:
    a = uk.groupby("age_year")[tcol].mean(); b = nl.groupby("age_year")[tcol].mean()
    j = pd.concat([a / a.mean(), b / b.mean()], axis=1, keys=["UK", "NL"])
    rows.append({"system": tcol, "age curve: Spearman (UK vs NL)": spearmanr(j.UK, j.NL)[0], "rate at age 5 / age 15, UK": a[5] / a[15], "NL": b[5] / b[15]})
display(pd.DataFrame(rows))

def make_oe(d, tcol):
    # expected = rate of the same age x fuel cell in that country; ratio observed/expected per make (makes with >= 1000 rows in both countries)
    exp = d.groupby(["age_year", "fuel"])[tcol].transform("mean")
    t = pd.DataFrame({"make": d.make_c, "o": d[tcol], "e": exp}).groupby("make").agg(o=("o", "sum"), e=("e", "sum"), n=("o", "size"))
    t["O/E"] = t.o / t.e; return t
rows = []
big = [m for m in COMMON_MAKES if (uk.make_c == m).sum() >= 1000 and (nl.make_c == m).sum() >= 1000]
for tcol in ["brakes", "electrical", "engine_emissions"]:
    a, b = make_oe(uk, tcol).loc[big], make_oe(nl, tcol).loc[big]
    rho, pv = spearmanr(a["O/E"], b["O/E"])
    rows.append({"system": tcol, "makes compared": len(big), "Spearman of make effect (O/E), UK vs NL": rho, "p-value": pv})
display(pd.DataFrame(rows))
mo = make_oe(uk, "brakes").loc[big, ["O/E"]].join(make_oe(nl, "brakes").loc[big, ["O/E"]], lsuffix=" UK", rsuffix=" NL")
print("brakes O/E by make (1.0 = as expected for its age and fuel), most different first:")
display(mo.assign(diff=(mo["O/E UK"] - mo["O/E NL"]).abs()).sort_values("diff", ascending=False).head(10))

,system,age curve: Spearman (UK vs NL),"rate at age 5 / age 15, UK",NL
0,brakes,0.9951,0.2529,0.2680
1,electrical,1.0000,0.1900,0.2845
2,engine_emissions,0.9951,0.1558,0.0749


,system,makes compared,"Spearman of make effect (O/E), UK vs NL",p-value
0,brakes,26,0.4509,0.0208
1,electrical,26,0.9467,0.0000
2,engine_emissions,26,0.6568,0.0003


brakes O/E by make (1.0 = as expected for its age and fuel), most different first:


,O/E UK,O/E NL,diff
make,,,
Dacia,1.4543,0.9405,0.5138
Renault,1.4467,1.0141,0.4327
Suzuki,1.2310,0.8714,0.3597
Land Rover,0.8868,1.1770,0.2901
Bmw,0.6616,0.9472,0.2856
Mitsubishi,1.2526,0.9862,0.2664
Audi,0.6460,0.9089,0.2628
Nissan,1.1513,0.9056,0.2457
Opel,1.2043,0.9610,0.2433


## 5. Weighted to look like the Israeli fleet
Test rows are re-weighted so that the (age, fuel, make-group) mix equals the Israeli sample. This answers "how would the ranking quality look on a fleet shaped like Israel's" **for the covariate mix only**; it does not fix differences in how defects are defined, tested or caused.

In [7]:
def il_weights(ted):
    cell = lambda d, mk: list(zip(d.age_year, d.fuel, d[mk]))
    il_cnt = pd.Series(cell(ilp.assign(make_c=ilp.make.where(ilp.make.isin(COMMON_MAKES), "OTHER")), "make_c")).value_counts(normalize=True)
    te_cnt = pd.Series(cell(ted, "make_c")).value_counts(normalize=True)
    return pd.Series(cell(ted, "make_c")).map(il_cnt / te_cnt).fillna(0.0).to_numpy()
rows = []
for ten, ted in TEST.items():
    w = il_weights(ted)
    for tname, tcol in TARGETS.items():
        y = ted[tcol].to_numpy()
        for trn in ["UK", "NL", "UK+NL pooled"]:
            p = PRED[(tname, "age, fuel, make", trn, ten)]
            rows.append({"tested on": ten, "system": tname, "trained on": trn, "AUC (own mix)": roc_auc_score(y, p), "AUC (Israeli mix)": roc_auc_score(y, p, sample_weight=w), "effective n": float(w.sum() ** 2 / (w ** 2).sum())})
display(pd.DataFrame(rows))

,tested on,system,trained on,AUC (own mix),AUC (Israeli mix),effective n
0,UK,brakes,UK,0.6766,0.6570,"12,393.8696"
1,UK,brakes,NL,0.6612,0.6506,"12,393.8696"
2,UK,brakes,UK+NL pooled,0.6730,0.6561,"12,393.8696"
3,UK,electrical,UK,0.7099,0.7006,"12,393.8696"
4,UK,electrical,NL,0.7020,0.6921,"12,393.8696"
5,UK,electrical,UK+NL pooled,0.7056,0.6961,"12,393.8696"
6,UK,engine-related,UK,0.7240,0.7399,"12,393.8696"
7,UK,engine-related,NL,0.6929,0.7304,"12,393.8696"
8,UK,engine-related,UK+NL pooled,0.7141,0.7332,"12,393.8696"
9,NL,brakes,UK,0.6488,0.6390,"22,949.3770"
